# Housing Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
housing_data = pd.read_csv('../data/Housing_Price_Data.csv')

In [ ]:
display(housing_data.head())

display(housing_data.info())

display(housing_data.describe())

display(housing_data.isnull().sum())

### **Get a sense of the data**

In [ ]:
fig = px.scatter(housing_data, 
                 x='area', 
                 y='price', 
                 color='furnishingstatus',
                 size='bedrooms',
                 hover_data=['bathrooms', 'stories'],
                 title='Price vs Area by Furnishing Status')
fig.show()

In [ ]:
fig = px.box(housing_data, 
             x='airconditioning', 
             y='price', 
             color='airconditioning',
             title='Effect of Air Conditioning on House Prices')
fig.show()


In [ ]:
fig = px.bar(housing_data.groupby(['furnishingstatus', 'parking'])['price'].mean().reset_index(),
             x='furnishingstatus',
             y='price',
             color='parking',
             barmode='group',
             title='Average House Price by Furnishing Status and Parking')
fig.show()


# Key Insights from Housing Price Visualizations

## 1. Price vs Area by Furnishing Status

**Visualization Type**: Bubble Scatter Plot

**Key Observations**:
- **Strong Positive Correlation**: House prices show a clear positive relationship with area across all furnishing statuses
- **Area Range**: Properties range from approximately 1,500 to 16,500 square units
- **Price Range**: Prices vary from ~₹2M to ~₹14M across the dataset
- **Furnishing Impact**: 
  - **Furnished** (Blue): Properties tend to cluster in the mid-to-high price range
  - **Semi-Furnished** (Red): Show mid-range pricing with moderate spread
  - **Unfurnished** (Green): Predominant category with widest distribution, generally lower prices for same area
- **Bedroom Distribution**: Bubble size indicates bedroom count, showing larger homes tend to be in furnished category
- **Market Concentration**: Most properties cluster between 4,000-8,000 square units with prices of 4M-7M

## 2. Effect of Air Conditioning on House Prices

**Visualization Type**: Box Plot Comparison

**Key Observations**:
- **Significant Price Premium**: Properties with air conditioning command higher prices
- **AC with Air Conditioning (Yes)**:
  - Median Price: ~₹6M
  - Price Range: ₹2M to ₹10M+
  - Notable outliers reaching up to ₹13M
  - More consistent pricing at higher values
  
- **AC Without Air Conditioning (No)**:
  - Median Price: ~₹4M
  - Price Range: ₹1.5M to ₹8M
  - Lower interquartile range
  - Fewer high-value outliers
  
- **Price Difference**: Properties with AC average ~₹2M higher than those without
- **Market Implication**: Air conditioning is a significant value-added feature in real estate market
- **Distribution**: Both groups show right-skewed distribution with presence of premium properties

## 3. Average House Price by Furnishing Status and Parking

**Visualization Type**: Stacked Bar Chart

**Key Observations**:
- **Furnishing Status Impact**:
  - **Furnished**: Highest average prices (~₹25M total), representing maximum value segment
  - **Semi-Furnished**: Mid-range prices (~₹20M total), balanced offering
  - **Unfurnished**: Lowest average prices (~₹18M total), budget-friendly segment
  
- **Parking Availability Effect**:
  - **High Parking (3+)**: Represented by yellow segments, commands premium
  - **Medium Parking (1-2)**: Orange segments, mid-level additional value
  - **Low Parking (0-1)**: Purple/Blue segments, base value
  
- **Correlation Insights**:
  - More parking spots consistently add value across all furnishing categories
  - Furnished homes with high parking achieve maximum value
  - Even unfurnished homes benefit significantly from parking availability
  
- **Market Segments**:
  - Furnished + High Parking: Premium segment (~₹10M-12M)
  - Semi-Furnished + Medium Parking: Mid-market segment (~₹8M-10M)
  - Unfurnished + Low Parking: Entry-level segment (~₹4M-6M)

---

## Summary of Key Market Drivers

| Factor | Impact | Priority |
|--------|--------|----------|
| **Area (Size)** | Strongest predictor - direct positive correlation | Critical |
| **Air Conditioning** | ~₹2M price premium (50% premium) | High |
| **Furnishing Status** | Furnished adds 30-40% premium vs unfurnished | High |
| **Parking Availability** | 3+ spots add significant premium | Medium-High |
| **Bedrooms** | Correlates with both area and price | Medium |

## Market Recommendations

1. **For Buyers**: 
   - Area is the primary value driver; focus on location and size
   - AC presence is highly valued; consider maintenance costs
   - Parking availability should influence location choice
   
2. **For Sellers**:
   - Improving furnishing status can increase appeal
   - Adding parking facilities provides good ROI
   - Air conditioning upgrade justified by 50% premium

3. **For Investors**:
   - Unfurnished properties in high-demand areas offer best ROI potential
   - Focus on properties with expansion/renovation opportunities
   - Market shows strong demand for both luxury and value segments

### Now we can make predictions on the house

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
housing_training_data = housing_data.copy()


def encode_dataset(df):
    df['furnishingstatus'] = df['furnishingstatus'].apply(lambda x: 2 if x == 'furnished' else (1 if x == 'semi-furnished' else 0))
    le = LabelEncoder()
    for col in ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']:
        df[col] = le.fit_transform(df[col])
    return df

housing_training_data = encode_dataset(housing_training_data)
X = housing_training_data.drop('price', axis=1)
y = housing_training_data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
base_tree = DecisionTreeRegressor(max_depth=4)
model = AdaBoostRegressor(estimator=base_tree, n_estimators=200, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
#  Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error: {mae}')
print(f'Mean Squared Error: {mse}')
print(f'R^2 Score: {r2}')

_omo_

In [ ]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_2 = StandardScaler()
X_train_2 = scaler_2.fit_transform(X_train_2)
X_test_2 = scaler_2.transform(X_test_2)

model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
model.fit(X_train_2, y_train_2)
y_pred_2 = model.predict(X_test_2)

In [ ]:
#  Evaluate the model
mae = mean_absolute_error(y_test_2, y_pred_2)
mse = mean_squared_error(y_test_2, y_pred_2)
r2 = r2_score(y_test_2, y_pred_2)

print(f'Mean Absolute Error: {mae}')
print(f'Mean Squared Error: {mse}')
print(f'R^2 Score: {r2}')

### **I feel, the dataset is a bit too small to make things work, theres one last option tho........**
___Basic linear regression___

In [ ]:
# using linear regression

from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

#  Evaluate the model
mae = mean_absolute_error(y_test, y_pred_linear)
mse = mean_squared_error(y_test, y_pred_linear)
r2 = r2_score(y_test, y_pred_linear)

print(f'Linear Regression - Mean Absolute Error: {mae}')
print(f'Linear Regression - Mean Squared Error: {mse}')
print(f'Linear Regression - R^2 Score: {r2}')

In [ ]:
# function to predict price based on input features
def predict_price(features):
    features_encoded = encode_dataset(pd.DataFrame([features]))
    features_scaled = scaler.transform(features_encoded)
    predicted_price = linear_model.predict(features_scaled)
    return predicted_price[0]

# Example usage
new_house = {
    'area': 3000,
    'bedrooms': 4,
    'bathrooms': 3,
    'stories': 2,
    'mainroad': 'yes',
    'guestroom': 'no',
    'basement': 'yes',
    'hotwaterheating': 'no',
    'airconditioning': 'yes',
    'parking': 2,
    'furnishingstatus': 'furnished',
    'prefarea': 'yes'
}

predicted_price = predict_price(new_house)
print(f'Predicted Price for the new house: {predicted_price}')